# Explorer 01 - Ingest and parse

**Purpose.** Read the eight gzipped XML dumps once and save them as normalized Parquet tables. Every other notebook reads the Parquet output and never touches the XML.

**Feeds main report section:** 3. Data and 4. Methodology (pipeline)

**Modules used:** `vinyl.config`, `vinyl.ingest`, `vinyl.tables`

**Tables read:** raw `discogs_<date>_<entity>.xml.gz` files

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

## Why loading is slow

1. **Schema inference.** Without a schema, Spark scans the whole file once to discover the columns, then again to load it. `ingest.read_raw_xml` passes the saved schemas in `schemas/`, which removes the first scan.
2. **Gzip is not splittable.** Each `.xml.gz` file is read by one task on one core, no matter how large the cluster is. The only cure is to do it once: `tables.build_all` stages the in-scope releases as Parquet and builds every child table from that copy.

Expect the releases files to take the longest. Run this notebook once per snapshot and leave it.

In [ ]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-01-ingest")
from vinyl import tables

## 1. Check paths and schemas

In [ ]:
for snap in config.SNAPSHOTS:
    for entity in config.ROW_TAGS:
        print(snap, entity, config.raw_path(snap, entity))

In [ ]:
# The saved schemas (from the October 2026 files) that replace inference.
for entity in config.ROW_TAGS:
    schema = ingest.load_schema(entity)
    print(f"{entity:9s} {len(schema.fields):2d} top-level fields: {schema.fieldNames()}")

## 2. Quick look at one small raw file

Labels is the smallest file, so it is a cheap test that the reader, row tag, and schema all work before starting the long releases job.

In [ ]:
raw_labels = ingest.read_raw_xml(spark, config.MAIN_SNAPSHOT, "labels")
raw_labels.printSchema()
raw_labels.show(5, truncate=60)

## 3. Build the Parquet tables

Start with the small tables to confirm the pipeline, then run the release tables. `build_all` reuses `releases_staged` if it already exists, so a failed run does not re-read the XML.

In [ ]:
tables.TABLES

In [ ]:
# Small tables first (minutes). fast=True (the default) splits each gz into
# plain-XML chunks so Spark parses them on all cores; the chunks are deleted
# after the Parquet write. 2020 builds labels, artists and masters (tables.snapshot_tables).
for snap in config.SNAPSHOTS:
    wanted = [t for t in tables.snapshot_tables(snap) if t in ("labels", "artists", "masters")]
    print(snap, tables.build_all(spark, snap, tables=wanted, fast=True, keep_stage=False))

In [ ]:
# Release tables (the long step). Run once per snapshot. fast=True splits the
# gz, drops releases dated outside 2016-2026 before parsing, and skips the
# unused fields in config.RELEASE_DROP_FIELDS. 2020 only builds the tables the
# snapshot comparison needs (tables.snapshot_tables).
for snap in config.SNAPSHOTS:
    wanted = [t for t in tables.snapshot_tables(snap) if t.startswith("release")]
    written = tables.build_all(spark, snap, tables=wanted, fast=True, keep_stage=False)
    for name, path in written.items():
        print(snap, name, path)

## 4. Sanity checks on the output

In [ ]:
# Row counts per table and snapshot. Tables a snapshot does not build
# (tables.snapshot_tables) show as NaN.
import pandas as pd

rows = []
for snap in config.SNAPSHOTS:
    for name in tables.snapshot_tables(snap):
        rows.append({"snapshot": snap, "table": name, "rows": ingest.read_table(spark, snap, name).count()})
pd.DataFrame(rows).pivot(index="table", columns="snapshot", values="rows").reindex(list(tables.TABLES))

In [ ]:
# Releases per year in the main snapshot: are all years 2016-2026 present?
releases = ingest.read_table(spark, config.MAIN_SNAPSHOT, "releases")
releases.groupBy("year").count().orderBy("year").show()

In [ ]:
# TODO: how many raw releases were dropped for having no usable `released` date?
# TODO: null rates of release_id, year, master_id, country.
# TODO: confirm the 2020 files parse with the 2026 schemas (one inferred read on a sample).

## Decisions to carry into `main.ipynb`

Filled in from the full-data runs in `executed/`. Each line should end up as a sentence in the report's methodology or limitations.

- [x] Number of releases in scope per snapshot: 3,829,109 in October 2026 and 1,082,351 in January 2020, all dated 2016-2026 (row counts in section 4).
- [ ] How many raw releases were dropped for missing dates - not decided: the year pre-filter runs while the XML is split, and its statistics were not kept for the full runs, so the count is not available (the `year_filter` row in main.ipynb 3.4 says so).
- [x] Whether the January 2020 files match the saved schemas: yes in practice; all 2020 tables (releases, release_labels, labels, artists, masters) were built with the saved October 2026 schemas without parse errors. The separate inferred-schema check on a sample (the third comment in the last code cell of section 4) was not run.
- [x] Spark version, cluster size, and wall-clock time: Spark 4.1.1 in local mode on one JupyterHub pod (4 CPUs, 16 GiB memory limit); the one-time conversion of both snapshots took 3 h 17 min (T03 run, `executed/README.md`).